# LeetCode 1673 — Find the Most Competitive Subsequence bằng Python

Link đề: https://leetcode.com/problems/find-the-most-competitive-subsequence/ · Độ khó: **Medium** · Chủ đề: Array, Stack, Greedy, Monotonic Stack

Skeleton LeetCode:

```python
class Solution:
    def mostCompetitive(self, nums: List[int], k: int) -> List[int]:
        ...
```

"Most competitive" thực chất là **dãy con độ dài `k` nhỏ nhất theo thứ tự từ điển**. Notebook đi từ vét cạn tổ hợp → tham lam theo cửa sổ `O(n·k)` → tham lam + heap `O(n log n)` → **ngăn xếp đơn điệu** `O(n)`.

## 1. Mục tiêu học

1. Hiểu **so sánh từ điển** (lexicographic) giữa hai dãy cùng độ dài và cách Python so sánh `list`/`tuple`.
2. Nắm lập luận tham lam: chọn phần tử **nhỏ nhất** có thể cho vị trí đầu tiên, nhưng phải **chừa đủ** phần tử cho các vị trí sau ⇒ "cửa sổ" hợp lệ.
3. Cài đặt tham lam theo cửa sổ (`O(n·k)`), rồi tăng tốc bằng heap (`O(n log n)`).
4. Thành thạo pattern **monotonic stack có giới hạn số lần pop**: `while stk and stk[-1] > x and còn_đủ_phần_tử: pop`.
5. Kiểm thử bằng vét cạn tổ hợp trên `n ≤ 12`, và đối chiếu chéo các cách trên mảng lớn hơn.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` và số nguyên dương `k`. Trả về **dãy con "cạnh tranh nhất"** của `nums` có độ dài `k`.

- **Dãy con** (subsequence) là dãy thu được bằng cách xóa một số (có thể không xóa) phần tử, **giữ nguyên thứ tự**.
- Dãy `a` **cạnh tranh hơn** dãy `b` (cùng độ dài) nếu ở **vị trí đầu tiên** mà `a`, `b` khác nhau, `a` có số **nhỏ hơn**. Ví dụ `[1,3,4]` cạnh tranh hơn `[1,3,5]` (khác ở vị trí cuối, `4 < 5`).

**Ví dụ 1:** `nums = [3,5,2,6]`, `k = 2` → `[2,6]`. Mọi dãy con độ dài 2: `{[3,5],[3,2],[3,6],[5,2],[5,6],[2,6]}`, nhỏ nhất là `[2,6]`.

**Ví dụ 2:** `nums = [2,4,3,3,5,4,9,6]`, `k = 4` → `[2,3,3,4]`.

**Ràng buộc:**
- `1 ≤ nums.length ≤ 10⁵`
- `0 ≤ nums[i] ≤ 10⁹`
- `1 ≤ k ≤ nums.length`

In [1]:
import random
import time
import heapq
from itertools import combinations
from typing import List

examples = [
    ([3, 5, 2, 6], 2, [2, 6]),
    ([2, 4, 3, 3, 5, 4, 9, 6], 4, [2, 3, 3, 4]),
]
for nums, k, out in examples:
    print(f"nums={nums}, k={k} -> kỳ vọng {out}")

nums=[3, 5, 2, 6], k=2 -> kỳ vọng [2, 6]
nums=[2, 4, 3, 3, 5, 4, 9, 6], k=4 -> kỳ vọng [2, 3, 3, 4]


## 3. Kiến thức nền

### 3.1 So sánh từ điển trong Python

Python so sánh `list` và `tuple` **theo từ điển**: so từng phần tử từ trái sang, phần tử khác nhau đầu tiên quyết định. Vì vậy "cạnh tranh hơn" chính là `a < b`, và `min(...)` trên một tập dãy cho dãy cạnh tranh nhất.

In [2]:
print([1, 3, 4] < [1, 3, 5], [2, 6] < [3, 2], (5, 2) < (3, 6))
print("min:", min([[3, 5], [3, 2], [3, 6], [5, 2], [5, 6], [2, 6]]))

True True False
min: [2, 6]


### 3.2 `itertools.combinations` — sinh mọi dãy con độ dài `k`

`combinations(range(n), k)` sinh mọi bộ chỉ số tăng dần độ dài `k` — mỗi bộ ứng với một dãy con. Có `C(n, k)` bộ: tăng rất nhanh (`C(20,10) = 184756`, `C(40,20) ≈ 1.4·10¹¹`).

In [3]:
nums = [3, 5, 2, 6]
for idx in combinations(range(len(nums)), 2):
    print(idx, "->", [nums[i] for i in idx])

(0, 1) -> [3, 5]
(0, 2) -> [3, 2]
(0, 3) -> [3, 6]
(1, 2) -> [5, 2]
(1, 3) -> [5, 6]
(2, 3) -> [2, 6]


### 3.3 Ngăn xếp bằng `list`

`append` (đẩy) và `pop()` (lấy đỉnh) đều `O(1)`; đỉnh là `stk[-1]`. **Ngăn xếp đơn điệu tăng**: trước khi đẩy `x`, bỏ mọi phần tử ở đỉnh **lớn hơn** `x`.

In [4]:
stk = []
for x in [3, 5, 2, 6, 4]:
    while stk and stk[-1] > x:
        print(f"  x={x}: pop {stk.pop()}")
    stk.append(x)
    print(f"đẩy {x} -> stk = {stk}")

đẩy 3 -> stk = [3]
đẩy 5 -> stk = [3, 5]
  x=2: pop 5
  x=2: pop 3
đẩy 2 -> stk = [2]
đẩy 6 -> stk = [2, 6]
  x=4: pop 6
đẩy 4 -> stk = [2, 4]


### 3.4 `heapq` — lấy phần tử nhỏ nhất nhanh

`heapq` là min-heap trên `list`: `heappush`, `heappop` `O(log n)`, `h[0]` là nhỏ nhất. Đẩy cặp `(giá trị, chỉ số)` thì khi giá trị bằng nhau, chỉ số **nhỏ hơn** được ưu tiên — đúng điều ta cần (chọn vị trí **trái nhất**).

In [5]:
h = []
for i, v in enumerate([4, 1, 3, 1]):
    heapq.heappush(h, (v, i))
print("nhỏ nhất:", h[0], "| lấy lần lượt:", [heapq.heappop(h) for _ in range(4)])

nhỏ nhất: (1, 1) | lấy lần lượt: [(1, 1), (1, 3), (3, 2), (4, 0)]


## 4. Ý tưởng nền tảng / trực giác — tính bằng tay

Ví dụ 2: `nums = [2,4,3,3,5,4,9,6]` (`n = 8`), `k = 4`.

**Vị trí đầu tiên** của đáp án nên nhỏ nhất có thể. Nhưng nếu chọn phần tử ở chỉ số `i` thì phía sau phải còn ít nhất `k − 1 = 3` phần tử ⇒ `i ≤ n − k = 4`. Trong `nums[0..4] = [2,4,3,3,5]`, nhỏ nhất là `2` (chỉ số 0).

**Vị trí thứ hai:** chọn sau chỉ số 0, và phải chừa 2 phần tử ⇒ chỉ số trong `[1, 5]`: `[4,3,3,5,4]` → nhỏ nhất `3`, **trái nhất** ở chỉ số 2.

**Vị trí thứ ba:** chỉ số trong `[3, 6]`: `[3,5,4,9]` → `3` (chỉ số 3).

**Vị trí thứ tư:** chỉ số trong `[4, 7]`: `[5,4,9,6]` → `4` (chỉ số 5).

Đáp án `[2,3,3,4]` ✓.

| vị trí j | cửa sổ chỉ số | giá trị trong cửa sổ | chọn |
|---|---|---|---|
| 0 | [0, 4] | 2 4 3 3 5 | 2 @0 |
| 1 | [1, 5] | 4 3 3 5 4 | 3 @2 |
| 2 | [3, 6] | 3 5 4 9 | 3 @3 |
| 3 | [4, 7] | 5 4 9 6 | 4 @5 |

Vì sao chọn **trái nhất** khi có nhiều giá trị nhỏ nhất bằng nhau? Vì chọn trái hơn thì cửa sổ sau **rộng hơn** (nhiều lựa chọn hơn) — không bao giờ thiệt.

In [6]:
def show_windows(nums, k):
    n, start = len(nums), 0
    res = []
    for j in range(k):
        end = n - k + j
        window = nums[start:end + 1]
        m = min(window)
        pos = start + window.index(m)        # index() trả vị trí trái nhất
        print(f"j={j}: cửa sổ [{start},{end}] = {window} -> chọn {m} @ {pos}")
        res.append(m)
        start = pos + 1
    return res

print(show_windows([2, 4, 3, 3, 5, 4, 9, 6], 4))

j=0: cửa sổ [0,4] = [2, 4, 3, 3, 5] -> chọn 2 @ 0
j=1: cửa sổ [1,5] = [4, 3, 3, 5, 4] -> chọn 3 @ 2
j=2: cửa sổ [3,6] = [3, 5, 4, 9] -> chọn 3 @ 3
j=3: cửa sổ [4,7] = [5, 4, 9, 6] -> chọn 4 @ 5
[2, 3, 3, 4]


## 5. Cách 1 — Brute Force: thử mọi dãy con độ dài `k`

**Ý tưởng:** sinh mọi bộ chỉ số bằng `combinations`, dựng dãy con, lấy `min`.

- Số dãy con `C(n, k)` ⇒ chỉ dùng được với `n` rất nhỏ (≈ 20). Đề có `n = 10⁵` ⇒ không thể.
- Nhưng đây là **chuẩn vàng** để kiểm tra các cách nhanh.

In [7]:
def competitive_brute(nums: List[int], k: int) -> List[int]:
    best = None
    for idx in combinations(range(len(nums)), k):
        cand = [nums[i] for i in idx]
        if best is None or cand < best:
            best = cand
    return best

for nums, k, out in examples:
    got = competitive_brute(nums, k)
    print(got)
    assert got == out

[2, 6]
[2, 3, 3, 4]


**Độ phức tạp Cách 1:** Time `O(C(n,k)·k)`, Space `O(k)`.

**Nút thắt:** số tổ hợp bùng nổ. Đo:

In [8]:
rnd = random.Random(1)
for n in (12, 16, 18):
    a = [rnd.randint(0, 9) for _ in range(n)]
    t = time.perf_counter(); competitive_brute(a, n // 2); dt = time.perf_counter() - t
    print(f"n={n}, k={n // 2}: C(n,k)={len(list(combinations(range(n), n // 2))):6}, {dt * 1000:7.1f} ms")

n=12, k=6: C(n,k)=   924,     0.2 ms
n=16, k=8: C(n,k)= 12870,     2.1 ms
n=18, k=9: C(n,k)= 48620,     9.1 ms


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (tham lam theo cửa sổ):** phần tử thứ `j` của đáp án (0-indexed) phải được chọn trong cửa sổ `[start, n − k + j]`, với `start` = ngay sau phần tử vừa chọn. Chọn **nhỏ nhất, trái nhất** trong cửa sổ luôn tối ưu (mục 10 chứng minh).

**Insight 2 (ngăn xếp):** nhìn theo hướng "xóa": ta phải xóa đúng `n − k` phần tử. Duyệt từ trái sang phải, giữ ngăn xếp là **ứng viên đáp án hiện tại**. Khi gặp `x`:
- Nếu đỉnh ngăn xếp `> x` **và** ta **vẫn còn quyền xóa** (tức sau khi xóa đỉnh, số phần tử trong ngăn xếp + số phần tử chưa duyệt vẫn ≥ `k`) **thì** xóa đỉnh — vì thay nó bằng `x` (nhỏ hơn) ở vị trí đó làm dãy nhỏ hơn theo từ điển.
- Sau đó, nếu ngăn xếp chưa đủ `k` phần tử **thì** đẩy `x`; nếu đã đủ thì bỏ qua `x` (đẩy vào chỉ làm dài quá `k`).

Điều kiện "còn quyền xóa" khi đang ở chỉ số `i`: `len(stk) − 1 + (n − i) ≥ k` ⇔ `len(stk) + n − i > k`.

**Invariant:** sau khi xử lý `nums[0..i]`, ngăn xếp là dãy con **nhỏ nhất theo từ điển** độ dài `min(k, …)` mà vẫn **có thể** kéo dài thành dãy con độ dài `k` bằng các phần tử phía sau.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Tham lam theo cửa sổ (`O(n·k)`)

**Từng đoạn:**
1. `start = 0`.
2. Với `j = 0..k−1`: cửa sổ `[start, n − k + j]`; tìm chỉ số trái nhất có giá trị nhỏ nhất.
3. Thêm giá trị đó vào đáp án, `start = pos + 1`.

Code demo đã có ở `show_windows` (mục 4). Code nộp:

In [9]:
class SolutionWindow:
    def mostCompetitive(self, nums: List[int], k: int) -> List[int]:
        n = len(nums)
        res = []
        start = 0
        for j in range(k):
            end = n - k + j
            pos = start
            for i in range(start + 1, end + 1):
                if nums[i] < nums[pos]:          # '<' nghiêm ngặt => giữ vị trí trái nhất
                    pos = i
            res.append(nums[pos])
            start = pos + 1
        return res

for nums, k, out in examples:
    assert SolutionWindow().mostCompetitive(nums, k) == out
print("Cách 2 qua ví dụ")

Cách 2 qua ví dụ


**Độ phức tạp Cách 2:** mỗi cửa sổ dài tối đa `n − k + 1`, có `k` cửa sổ ⇒ Time `O(k·(n − k + 1))` — tệ nhất `O(n²/4)` khi `k ≈ n/2`. Space `O(k)` cho đáp án. Với `n = 10⁵` vẫn TLE.

### 7.2 Cách 3 — Tham lam theo cửa sổ + heap (`O(n log n)`)

Nhận xét: đầu phải cửa sổ `n − k + j` **tăng dần 1** mỗi bước, đầu trái `start` cũng chỉ tăng. Vậy:
1. Đẩy dần `(nums[i], i)` vào heap khi đầu phải mở rộng tới `i`.
2. Đỉnh heap có chỉ số `< start` là "hết hạn" ⇒ bỏ đi (**xóa lười** — lazy deletion).
3. Đỉnh còn lại là `(giá trị nhỏ nhất, chỉ số trái nhất)` trong cửa sổ.

Code demo:

In [10]:
def demo_heap(nums, k):
    n = len(nums)
    h, res, start, nxt = [], [], 0, 0
    for j in range(k):
        end = n - k + j
        while nxt <= end:
            heapq.heappush(h, (nums[nxt], nxt)); nxt += 1
        dropped = []
        while h[0][1] < start:
            dropped.append(heapq.heappop(h))
        v, pos = heapq.heappop(h)
        print(f"j={j}: cửa sổ [{start},{end}], bỏ hết hạn {dropped}, chọn ({v}, @{pos})")
        res.append(v); start = pos + 1
    return res

print(demo_heap([2, 4, 3, 3, 5, 4, 9, 6], 4))

j=0: cửa sổ [0,4], bỏ hết hạn [], chọn (2, @0)
j=1: cửa sổ [1,5], bỏ hết hạn [], chọn (3, @2)
j=2: cửa sổ [3,6], bỏ hết hạn [], chọn (3, @3)
j=3: cửa sổ [4,7], bỏ hết hạn [(4, 1)], chọn (4, @5)
[2, 3, 3, 4]


Code nộp LeetCode của Cách 3:

In [11]:
class SolutionHeap:
    def mostCompetitive(self, nums: List[int], k: int) -> List[int]:
        n = len(nums)
        h = []
        res = []
        start = nxt = 0
        for j in range(k):
            end = n - k + j
            while nxt <= end:
                heapq.heappush(h, (nums[nxt], nxt))
                nxt += 1
            while h[0][1] < start:           # xóa lười phần tử nằm trước cửa sổ
                heapq.heappop(h)
            v, pos = heapq.heappop(h)
            res.append(v)
            start = pos + 1
        return res

for nums, k, out in examples:
    assert SolutionHeap().mostCompetitive(nums, k) == out
print("Cách 3 qua ví dụ")

Cách 3 qua ví dụ


**Độ phức tạp Cách 3:** mỗi chỉ số được đẩy và lấy ra tối đa một lần ⇒ Time `O(n log n)`, Space `O(n)`.

### 7.3 Cách 4 — Ngăn xếp đơn điệu có giới hạn (`O(n)`)

**Từng đoạn:**
1. `stk = []`.
2. Với mỗi `i, x`: trong khi `stk` khác rỗng, `stk[-1] > x` **và** `len(stk) + (n − i) > k` ⇒ `stk.pop()`.
3. Nếu `len(stk) < k` ⇒ `stk.append(x)`.
4. Trả `stk`.

Code demo (in ngăn xếp và "số lượt xóa còn lại" `n − k − (số phần tử đã bỏ)`):

In [12]:
def demo_stack(nums, k):
    n = len(nums)
    stk = []
    for i, x in enumerate(nums):
        while stk and stk[-1] > x and len(stk) + n - i > k:
            print(f"   i={i}, x={x}: pop {stk.pop()}")
        if len(stk) < k:
            stk.append(x); act = "đẩy"
        else:
            act = "bỏ qua"
        removed = i + 1 - len(stk)
        print(f"i={i}, x={x}: {act:6} stk={stk}  (đã bỏ {removed}/{n - k})")
    return stk

print(demo_stack([3, 5, 2, 6], 2))

i=0, x=3: đẩy    stk=[3]  (đã bỏ 0/2)
i=1, x=5: đẩy    stk=[3, 5]  (đã bỏ 0/2)
   i=2, x=2: pop 5
   i=2, x=2: pop 3
i=2, x=2: đẩy    stk=[2]  (đã bỏ 2/2)
i=3, x=6: đẩy    stk=[2, 6]  (đã bỏ 2/2)
[2, 6]


Code nộp LeetCode của Cách 4:

In [13]:
class SolutionStack:
    def mostCompetitive(self, nums: List[int], k: int) -> List[int]:
        n = len(nums)
        stk = []
        for i, x in enumerate(nums):
            while stk and stk[-1] > x and len(stk) + n - i > k:
                stk.pop()
            if len(stk) < k:
                stk.append(x)
        return stk

for nums, k, out in examples:
    assert SolutionStack().mostCompetitive(nums, k) == out
print("Cách 4 qua ví dụ")

Cách 4 qua ví dụ


**Độ phức tạp Cách 4:** mỗi phần tử được đẩy tối đa một lần và lấy ra tối đa một lần ⇒ Time `O(n)` (phân tích khấu hao), Space `O(k)`.

Biến thể tương đương: đếm `drop = n − k` "lượt xóa"; pop khi `drop > 0` rồi `drop -= 1`; cuối cùng trả `stk[:k]`.

## 8. TRACE từng bước

Trace Cách 4 trên ví dụ 2: `nums = [2,4,3,3,5,4,9,6]`, `k = 4`, `n = 8` (được xóa `4` phần tử).

| i | x | điều kiện pop `len+n−i>k` | thao tác | stk sau |
|---|---|---|---|---|
| 0 | 2 | — | đẩy | [2] |
| 1 | 4 | — | đẩy | [2,4] |
| 2 | 3 | 4>3, `2+6=8>4` ✓ | pop 4, đẩy 3 | [2,3] |
| 3 | 3 | 3>3? không | đẩy | [2,3,3] |
| 4 | 5 | — | đẩy | [2,3,3,5] |
| 5 | 4 | 5>4, `4+3=7>4` ✓ | pop 5, đẩy 4 | [2,3,3,4] |
| 6 | 9 | không pop; đã đủ k | bỏ qua | [2,3,3,4] |
| 7 | 6 | 4>6? không; đủ k | bỏ qua | [2,3,3,4] |

In [14]:
def trace_stack(nums, k):
    n = len(nums)
    stk = []
    print(" i |  x | pop                 | thao tác | stk")
    for i, x in enumerate(nums):
        popped = []
        while stk and stk[-1] > x and len(stk) + n - i > k:
            popped.append(stk.pop())
        if len(stk) < k:
            stk.append(x); act = "đẩy"
        else:
            act = "bỏ qua"
        print(f" {i} | {x:2} | {str(popped):19} | {act:8} | {stk}")
    return stk

assert trace_stack([2, 4, 3, 3, 5, 4, 9, 6], 4) == [2, 3, 3, 4]

 i |  x | pop                 | thao tác | stk
 0 |  2 | []                  | đẩy      | [2]
 1 |  4 | []                  | đẩy      | [2, 4]
 2 |  3 | [4]                 | đẩy      | [2, 3]
 3 |  3 | []                  | đẩy      | [2, 3, 3]
 4 |  5 | []                  | đẩy      | [2, 3, 3, 5]
 5 |  4 | [5]                 | đẩy      | [2, 3, 3, 4]
 6 |  9 | []                  | bỏ qua   | [2, 3, 3, 4]
 7 |  6 | []                  | bỏ qua   | [2, 3, 3, 4]


Trace thêm một trường hợp mà điều kiện "còn quyền xóa" **chặn** việc pop: `nums = [5,4,3,2,1]`, `k = 3` (ví dụ tự dựng, đã kiểm bằng brute force).

In [15]:
res = trace_stack([5, 4, 3, 2, 1], 3)
print("brute force:", competitive_brute([5, 4, 3, 2, 1], 3))
assert res == competitive_brute([5, 4, 3, 2, 1], 3)

 i |  x | pop                 | thao tác | stk
 0 |  5 | []                  | đẩy      | [5]
 1 |  4 | [5]                 | đẩy      | [4]
 2 |  3 | [4]                 | đẩy      | [3]
 3 |  2 | []                  | đẩy      | [3, 2]
 4 |  1 | []                  | đẩy      | [3, 2, 1]
brute force: [3, 2, 1]


## 9. Code nộp LeetCode

In [16]:
class Solution:
    def mostCompetitive(self, nums: List[int], k: int) -> List[int]:
        n = len(nums)
        stk = []
        for i, x in enumerate(nums):
            # còn đủ phần tử phía sau để bù chỗ trống sau khi pop?
            while stk and stk[-1] > x and len(stk) + n - i > k:
                stk.pop()
            if len(stk) < k:
                stk.append(x)
        return stk

for nums, k, out in examples:
    got = Solution().mostCompetitive(nums, k)
    print(got)
    assert got == out

[2, 6]
[2, 3, 3, 4]


## 10. Vì sao thuật toán đúng

**Tham lam theo cửa sổ (Cách 2, 3).** Gọi `p` là chỉ số được chọn cho vị trí `j`. Mọi dãy con hợp lệ phải đặt phần tử thứ `j` ở một chỉ số trong cửa sổ `[start, n − k + j]` (để còn đủ `k − 1 − j` phần tử phía sau). Nếu một đáp án đặt ở vị trí `j` một giá trị lớn hơn `min` của cửa sổ thì nó thua ngay tại vị trí `j`. Trong các chỉ số đạt `min`, chọn trái nhất `p`: mọi tiếp nối khả dĩ sau một chỉ số `p' > p` cũng là tiếp nối khả dĩ sau `p` (tập phần tử phía sau lớn hơn) ⇒ không thiệt. Quy nạp theo `j`.

**Ngăn xếp (Cách 4).** Hai tính chất:
1. **Không bao giờ thiếu phần tử:** chỉ pop khi `len(stk) − 1 + (n − i) ≥ k`, nên luôn còn đủ phần tử để lấp đầy `k` ⇒ kết quả có đúng `k` phần tử.
2. **Pop là có lợi:** nếu đỉnh `t > x` và vẫn được phép xóa, thì dãy có `x` ở vị trí của `t` nhỏ hơn mọi dãy giữ `t` ở vị trí đó (vì các vị trí trước giống nhau). Ngược lại, không pop khi `t ≤ x` vì thay `t` bằng `x` không làm nhỏ hơn.

Kết quả ngăn xếp trùng với kết quả tham lam theo cửa sổ: mỗi phần tử còn lại trong ngăn xếp là phần tử nhỏ nhất, trái nhất có thể ở vị trí của nó — điều mà các test ngẫu nhiên ở mục 12 kiểm chứng.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`.

| Cách | Time | Space phụ |
|---|---|---|
| 1. Vét cạn | `O(C(n,k)·k)` | `O(k)` |
| 2. Cửa sổ | `O(k·(n−k+1))`, tệ nhất `O(n²)` | `O(k)` |
| 3. Cửa sổ + heap | `O(n log n)` | `O(n)` |
| 4. Ngăn xếp | `O(n)` khấu hao | `O(k)` |

Ngăn xếp `O(n)`: vòng `while` có thể chạy nhiều lần trong một bước, nhưng **tổng** số lần pop qua cả thuật toán ≤ số lần đẩy ≤ `n`.

In [17]:
rnd = random.Random(7)
big = [rnd.randint(0, 10**9) for _ in range(10**5)]
for name, sol, n_use in [("Cách 2 (n=4000)", SolutionWindow(), 4000), ("Cách 3", SolutionHeap(), 10**5),
                         ("Cách 4", SolutionStack(), 10**5)]:
    a = big[:n_use]
    t = time.perf_counter(); r = sol.mostCompetitive(a, len(a) // 2); dt = time.perf_counter() - t
    print(f"{name:16}: {dt * 1000:8.1f} ms, độ dài đáp án {len(r)}")

Cách 2 (n=4000) :      0.3 ms, độ dài đáp án 2000
Cách 3          :     59.6 ms, độ dài đáp án 50000
Cách 4          :      9.9 ms, độ dài đáp án 50000


## 12. Bộ test

In [18]:
FAST = [("window", SolutionWindow().mostCompetitive), ("heap", SolutionHeap().mostCompetitive),
        ("stack", SolutionStack().mostCompetitive), ("submit", Solution().mostCompetitive)]

def check(nums, k, expected=None):
    ref = competitive_brute(nums, k) if expected is None else expected
    for name, f in FAST:
        got = f(list(nums), k)
        assert got == ref, (name, nums, k, got, ref)

for nums, k, out in examples:
    check(nums, k, out)
# edge case (tự dựng, đối chiếu brute force)
for nums, k in [([7], 1), ([1, 2, 3], 3), ([3, 2, 1], 3), ([3, 2, 1], 1), ([5, 4, 3, 2, 1], 3),
                ([1, 1, 1, 1], 2), ([0, 10**9, 0], 2), ([2, 1, 2, 1, 2], 3), ([1, 1, 2], 2)]:
    check(nums, k)
print("ví dụ đề + edge case OK")

ví dụ đề + edge case OK


In [19]:
# ngẫu nhiên nhỏ: đối chiếu vét cạn (seed cố định, đúng ràng buộc)
rnd = random.Random(2024)
for _ in range(1500):
    n = rnd.randint(1, 11)
    k = rnd.randint(1, n)
    hi = rnd.choice([1, 3, 10, 10**9])
    nums = [rnd.randint(0, hi) for _ in range(n)]
    check(nums, k)
# ngẫu nhiên lớn: đối chiếu chéo các cách nhanh
for _ in range(30):
    n = rnd.randint(1, 1500); k = rnd.randint(1, n)
    nums = [rnd.randint(0, rnd.choice([5, 10**9])) for _ in range(n)]
    outs = [f(nums, k) for _, f in FAST]
    assert all(o == outs[0] for o in outs) and len(outs[0]) == k
print("test ngẫu nhiên OK")

test ngẫu nhiên OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Pop không kiểm tra "còn đủ phần tử"

In [20]:
def wrong_no_limit(nums, k):
    stk = []
    for x in nums:
        while stk and stk[-1] > x:      # SAI: thiếu điều kiện len(stk) + n - i > k
            stk.pop()
        if len(stk) < k:
            stk.append(x)
    return stk

nums, k = [5, 4, 3, 2, 1], 3
print("sai :", wrong_no_limit(nums, k), "| đúng:", Solution().mostCompetitive(nums, k))

sai : [1] | đúng: [3, 2, 1]


**Vì sao sai:** dãy giảm dần làm ngăn xếp bị xóa sạch liên tục, cuối cùng chỉ còn `[1]` — thiếu phần tử (độ dài < `k`). **Sửa:** chỉ pop khi `len(stk) + n − i > k`.

### Lỗi 2 — Luôn đẩy phần tử, quên giới hạn `len(stk) < k`

In [21]:
def wrong_always_push(nums, k):
    n = len(nums); stk = []
    for i, x in enumerate(nums):
        while stk and stk[-1] > x and len(stk) + n - i > k:
            stk.pop()
        stk.append(x)                    # SAI: có thể vượt quá k phần tử
    return stk

nums, k = [3, 5, 2, 6], 2
print("sai :", wrong_always_push(nums, k), "| đúng:", Solution().mostCompetitive(nums, k))

sai : [2, 6] | đúng: [2, 6]


**Vì sao sai:** khi các phần tử cuối lớn hơn đỉnh, chúng vẫn được đẩy vào ⇒ dãy dài hơn `k`. **Sửa:** chỉ đẩy khi `len(stk) < k` (hoặc trả `stk[:k]` — cũng đúng vì phần đầu ngăn xếp đã tối ưu).

### Lỗi 3 — Chọn phần tử nhỏ nhất **toàn cục** trước

In [22]:
def wrong_global_min(nums, k):
    i = nums.index(min(nums))            # SAI: có thể không còn đủ phần tử phía sau
    rest = nums[i:]
    return (rest + [None] * k)[:k]

nums, k = [3, 5, 2, 6], 3
print("sai :", wrong_global_min(nums, k), "| đúng:", Solution().mostCompetitive(nums, k))

sai : [2, 6, None] | đúng: [3, 2, 6]


**Vì sao sai:** `2` ở chỉ số 2 nhưng phía sau chỉ còn 1 phần tử, không đủ tạo dãy độ dài 3. Phải chọn trong **cửa sổ** `[0, n − k]`. **Sửa:** tham lam theo cửa sổ hoặc ngăn xếp có giới hạn.

### Lỗi 4 — Khi có nhiều giá trị nhỏ nhất, chọn vị trí **phải nhất**

In [23]:
def wrong_rightmost(nums, k):
    n = len(nums); res = []; start = 0
    for j in range(k):
        end = n - k + j
        pos = start
        for i in range(start + 1, end + 1):
            if nums[i] <= nums[pos]:     # SAI: '<=' chọn vị trí phải nhất
                pos = i
        res.append(nums[pos]); start = pos + 1
    return res

nums, k = [1, 1, 2], 2
print("sai :", wrong_rightmost(nums, k), "| đúng:", Solution().mostCompetitive(nums, k))

sai : [1, 2] | đúng: [1, 1]


**Vì sao sai:** chọn `1` ở chỉ số 1 làm mất cơ hội dùng `1` ở chỉ số 1 cho vị trí tiếp theo, nên vị trí 2 chỉ còn `2`. **Sửa:** so sánh `<` nghiêm ngặt (giữ vị trí trái nhất); với heap dùng cặp `(giá trị, chỉ số)`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Dãy con độ dài `k` nhỏ/lớn nhất theo từ điển", "xóa `k` chữ số để được số nhỏ nhất" ⇒ **monotonic stack + đếm số lượt xóa**.
- Mẫu điều kiện: `while stk and stk[-1] > x and (còn lượt xóa)`; sau đó `if len(stk) < k: append`.
- Hai cách viết "còn lượt xóa": `len(stk) + n − i > k` hoặc biến `drop = n − k` giảm dần.
- Muốn **lớn nhất** theo từ điển thì đổi `>` thành `<`.
- Khi nghi ngờ, viết tham lam theo cửa sổ — đúng hiển nhiên — và đối chiếu với ngăn xếp.

## 15. Biến thể của bài

1. **LC 402 — Remove K Digits:** xóa `k` chữ số để số nhỏ nhất (thêm xử lý số 0 ở đầu) — bài tập 5.
2. **LC 316 / 1081 — Remove Duplicate Letters / Smallest Subsequence of Distinct Characters:** ngăn xếp đơn điệu + ràng buộc "còn xuất hiện phía sau".
3. **LC 321 — Create Maximum Number:** dãy con độ dài `k` **lớn nhất** của từng mảng rồi trộn.
4. **LC 2030 — Smallest K-Length Subsequence With Occurrences of a Letter:** thêm ràng buộc số lần xuất hiện của một ký tự.
5. **Dãy con lớn nhất theo từ điển** độ dài `k` (bài tập 4).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | `min` trên mọi tổ hợp | `O(C(n,k)·k)` | `O(k)` | kiểm chứng `n ≤ 15` |
| 2. Cửa sổ | nhỏ nhất, trái nhất trong `[start, n−k+j]` | `O(n·k)` | `O(k)` | dễ chứng minh |
| 3. Cửa sổ + heap | heap `(v, i)` + xóa lười | `O(n log n)` | `O(n)` | khi chưa nghĩ ra ngăn xếp |
| 4. Ngăn xếp đơn điệu | pop khi đỉnh lớn hơn và còn lượt xóa | `O(n)` | `O(k)` | **nộp bài** |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** (a) `nums = [1,2,3]`, `k = 2`; (b) `nums = [3,2,1]`, `k = 2`; (c) `nums = [5,5,4,4]`, `k = 2`.

**Bài 2 (tính tay):** Với `nums = [9,1,8,2,7,3]`, `k = 3`, ghi ngăn xếp sau mỗi bước của Cách 4.

**Bài 3 (điền TODO):** Hoàn thiện `competitive_drop(nums, k)` theo kiểu dùng biến `drop = n − k`.

**Bài 4 (tự viết lại):** Viết `max_subsequence(nums, k)` trả dãy con độ dài `k` **lớn nhất** theo từ điển.

**Bài 5 (mini challenge):** LC 402 — `remove_k_digits(num: str, k: int) -> str`: xóa đúng `k` chữ số để được số nhỏ nhất; bỏ số 0 ở đầu; nếu rỗng trả `"0"`.

In [24]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def competitive_drop(nums, k):
    drop = len(nums) - k
    stk = []
    for x in nums:
        # TODO: pop khi đỉnh > x và còn drop
        stk.append(x)
    # TODO: cắt về đúng k phần tử
    return stk

In [25]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def max_subsequence(nums, k):
    # TODO
    pass

def remove_k_digits(num, k):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: cửa sổ đầu là `[0, n−k]`. Mức 2: (c) cửa sổ đầu `[5,5,4]` → `4` @2. Mức 3: (a) `[1,2]`; (b) `[2,1]`; (c) `[4,4]`.

**Bài 2.** Mức 1: được xóa `3` phần tử. Mức 2: `[9]` → `[1]` → `[1,8]` → `[1,2]` → `[1,2,7]`. Mức 3: ở `x = 3`, đỉnh `7 > 3` và `3 + 1 > 3` ⇒ pop ⇒ `[1,2,3]`.

**Bài 3.** Mức 1: `while drop and stk and stk[-1] > x`. Mức 2: mỗi lần pop thì `drop -= 1`. Mức 3: `return stk[:k]`.

**Bài 4.** Mức 1: đối xứng với bài gốc. Mức 2: đổi `stk[-1] > x` thành `stk[-1] < x`. Mức 3: đối chiếu với `max` trên `combinations`.

**Bài 5.** Mức 1: giống Bài 3 trên chuỗi chữ số, giữ `len(num) − k` chữ số. Mức 2: `"".join(stk).lstrip("0")`. Mức 3: trả `"0"` nếu rỗng; kiểm tra bằng vét cạn `combinations` + `int(...)`.

## 19. Đáp án tham khảo cho bài tập

In [26]:
# Bài 1, 2
b1 = [Solution().mostCompetitive(a, k) for a, k in (([1, 2, 3], 2), ([3, 2, 1], 2), ([5, 5, 4, 4], 2))]
print("Bài 1:", b1)
assert b1 == [[1, 2], [2, 1], [4, 4]]
res = trace_stack([9, 1, 8, 2, 7, 3], 3)
assert res == [1, 2, 3] == competitive_brute([9, 1, 8, 2, 7, 3], 3)

Bài 1: [[1, 2], [2, 1], [4, 4]]
 i |  x | pop                 | thao tác | stk
 0 |  9 | []                  | đẩy      | [9]
 1 |  1 | [9]                 | đẩy      | [1]
 2 |  8 | []                  | đẩy      | [1, 8]
 3 |  2 | [8]                 | đẩy      | [1, 2]
 4 |  7 | []                  | đẩy      | [1, 2, 7]
 5 |  3 | [7]                 | đẩy      | [1, 2, 3]


In [27]:
# Bài 3
def competitive_drop_sol(nums, k):
    drop = len(nums) - k
    stk = []
    for x in nums:
        while drop and stk and stk[-1] > x:
            stk.pop()
            drop -= 1
        stk.append(x)
    return stk[:k]

# Bài 4
def max_subsequence_sol(nums, k):
    n = len(nums); stk = []
    for i, x in enumerate(nums):
        while stk and stk[-1] < x and len(stk) + n - i > k:
            stk.pop()
        if len(stk) < k:
            stk.append(x)
    return stk

def max_brute(nums, k):
    return max([nums[i] for i in idx] for idx in combinations(range(len(nums)), k))

rnd = random.Random(55)
for _ in range(1500):
    n = rnd.randint(1, 10); k = rnd.randint(1, n)
    a = [rnd.randint(0, 5) for _ in range(n)]
    assert competitive_drop_sol(a, k) == competitive_brute(a, k)
    assert max_subsequence_sol(a, k) == max_brute(a, k)
print("Bài 3, 4 OK")

Bài 3, 4 OK


In [28]:
# Bài 5 (LC 402)
def remove_k_digits_sol(num, k):
    stk = []
    drop = k
    for c in num:
        while drop and stk and stk[-1] > c:
            stk.pop(); drop -= 1
        stk.append(c)
    stk = stk[:len(num) - k]
    return "".join(stk).lstrip("0") or "0"

def remove_k_brute(num, k):
    keep = len(num) - k
    if keep == 0:
        return "0"
    return str(min(int("".join(num[i] for i in idx)) for idx in combinations(range(len(num)), keep)))

for num, k, exp in [("1432219", 3, "1219"), ("10200", 1, "200"), ("10", 2, "0")]:
    assert remove_k_digits_sol(num, k) == exp
rnd = random.Random(56)
for _ in range(1500):
    n = rnd.randint(1, 9)
    num = str(rnd.randint(1, 9)) + "".join(rnd.choice("0123456789") for _ in range(n - 1))
    k = rnd.randint(0, n)
    assert remove_k_digits_sol(num, k) == remove_k_brute(num, k), (num, k)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "dãy con độ dài k nhỏ nhất theo từ điển":**

```python
n, stk = len(nums), []
for i, x in enumerate(nums):
    while stk and stk[-1] > x and len(stk) + n - i > k:   # còn đủ phần tử để bù
        stk.pop()
    if len(stk) < k:
        stk.append(x)
return stk
```

(Lớn nhất theo từ điển: đổi `>` thành `<` ở `stk[-1] > x`.)

**Checklist:**
- [ ] Điều kiện pop có **cả** `stk[-1] > x` **và** `len(stk) + n − i > k`?
- [ ] Chỉ đẩy khi `len(stk) < k` (hoặc cắt `[:k]` cuối cùng)?
- [ ] So sánh `>` nghiêm ngặt (không pop phần tử bằng `x` — không cần thiết)?
- [ ] Kết quả có đúng `k` phần tử trên dãy giảm dần, dãy tăng dần, dãy hằng?
- [ ] Đã đối chiếu với vét cạn `combinations` trên `n ≤ 11`?